In [ ]:
import cv2
import numpy as np
import joblib
from ultralytics import YOLO

#loading models
pose_model = YOLO('yolov8n-pose.pt')
classifier = joblib.load('model.pkl')

#keypoints to drop same as in training
drop_kps = [13,14,15,16]
drop_indices = []
for i in drop_kps:
    drop_indices += [i*3, i*3+1, i*3+2]

print("Models loaded.")
print(f"Dropping feature indices: {drop_indices}")


In [1]:
cap = cv2.VideoCapture(0)
print("Press 'q' to quit")

while cap.isOpened():
    ret,frame = cap.read()
    if not ret:
        break

    results = pose_model(frame, verbose=False)
    annotated = results[0].plot()

    if results[0].keypoints is not None and len(results[0].keypoints.data) > 0:
        kps = results[0].keypoints.data[0].cpu().numpy().flatten()
        features = np.delete(kps, drop_indices).reshape(1,-1)
        prediction = classifier.predict(features)[0]

        label = "GOOD POSTURE" if prediction == 0 else "SLOUCH"
        color = (0,255,0) if prediction == 0 else (0,0,255)
    else:
        label = "No person detected"
        color = (255,255,255)

    cv2.putText(annotated, label, (30,60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, color, 3)

    cv2.imshow('Posture Monitor', annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


NameError: name 'cv2' is not defined